# 🟩 Celda 1 – Imports y configuración inicial

In [1]:
# 03_train_xgboost.ipynb
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb
import optuna

# Paths
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
MODELS_DIR = os.path.join(BASE_DIR, "models")
RESULTS_DIR = os.path.join(BASE_DIR, "results")

os.makedirs(RESULTS_DIR, exist_ok=True)

print("📁 Paths configurados correctamente")


📁 Paths configurados correctamente


c:\Users\yeder\Documents\Factoria F5 Bootcamp IA\proyecto7_ensemble_grupo2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 🟩 Celda 2 – Cargar embeddings

In [3]:
data = joblib.load(os.path.join(MODELS_DIR, "embeddings.joblib"))

X = data["embeddings"]
y = data["y"]
class_names = data["class_names"]

print("Embeddings shape:", X.shape)
print("Etiquetas únicas:", len(np.unique(y)))


Embeddings shape: (20960, 512)
Etiquetas únicas: 21


# 🟩 Celda 3 – Dividir datos en folds para validación cruzada

In [4]:
from sklearn.preprocessing import LabelEncoder

# Codifica etiquetas si no lo están
le = LabelEncoder()
y_enc = le.fit_transform(y)

# Definimos los folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("✅ StratifiedKFold listo (5 folds)")


✅ StratifiedKFold listo (5 folds)


# 🟩 Celda 4 – Definir función de entrenamiento y evaluación

In [5]:
def train_xgboost_fold(X_train, y_train, X_val, y_val, params):
    model = xgb.XGBClassifier(
        **params,
        objective="multi:softprob",
        num_class=len(np.unique(y_train)),
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)],
              verbose=False)
    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)
    return model, acc, preds


# 🟩 Celda 5 – Definir parámetros base de XGBoost

In [6]:
params = {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "gamma": 1,
}


# 🟩 Celda 6 – Entrenamiento con validación cruzada

In [7]:
accuracies = []
fold = 1
best_models = []

for train_idx, val_idx in skf.split(X, y_enc):
    print(f"\n🔹 Fold {fold}")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y_enc[train_idx], y_enc[val_idx]

    model, acc, preds = train_xgboost_fold(X_train, y_train, X_val, y_val, params)
    print(f"Accuracy: {acc:.4f}")

    accuracies.append(acc)
    best_models.append(model)
    fold += 1

print("\n✅ Mean accuracy:", np.mean(accuracies))



🔹 Fold 1
Accuracy: 0.4318

🔹 Fold 2
Accuracy: 0.4275

🔹 Fold 3
Accuracy: 0.4292

🔹 Fold 4
Accuracy: 0.4327

🔹 Fold 5
Accuracy: 0.4308

✅ Mean accuracy: 0.4303912213740458


# 

# 🟩 Celda 7 – Guardar mejor modelo y métricas

In [8]:
best_model = best_models[np.argmax(accuracies)]
out_model_path = os.path.join(MODELS_DIR, "xgb_final.joblib")
joblib.dump(best_model, out_model_path)

metrics_path = os.path.join(RESULTS_DIR, "metrics.csv")
pd.DataFrame({"fold": list(range(1,6)), "accuracy": accuracies}).to_csv(metrics_path, index=False)

print("💾 Modelo guardado en:", out_model_path)
print("📊 Métricas guardadas en:", metrics_path)


💾 Modelo guardado en: c:\Users\yeder\Documents\Factoria F5 Bootcamp IA\proyecto7_ensemble_grupo2\backend\ML\models\xgb_final.joblib
📊 Métricas guardadas en: c:\Users\yeder\Documents\Factoria F5 Bootcamp IA\proyecto7_ensemble_grupo2\backend\ML\results\metrics.csv


# 🟩 Celda 8 (opcional) – Reporte final

In [9]:
y_pred = best_model.predict(X)
print("🔍 Reporte global:")
print(classification_report(y_enc, y_pred, target_names=class_names))


🔍 Reporte global:
                         precision    recall  f1-score   support

              apple_pie       0.88      0.84      0.86       998
               beignets       0.90      0.92      0.91       997
          bread_pudding       0.87      0.87      0.87       996
      breakfast_burrito       0.88      0.86      0.87       999
                cannoli       0.90      0.90      0.90       998
            carrot_cake       0.90      0.88      0.89       998
             cheesecake       0.89      0.87      0.88       998
         chocolate_cake       0.90      0.90      0.90      1000
                churros       0.90      0.90      0.90       979
          club_sandwich       0.90      0.92      0.91       999
          croque_madame       0.91      0.89      0.90       999
              cup_cakes       0.91      0.92      0.91       999
                 donuts       0.91      0.91      0.91      1000
          eggs_benedict       0.89      0.91      0.90      1000
      